# Pulse calibration workflow

This notebook calibrates the default half-pi / pi pulses first, then
shows how to refine the DRAG versions. The goal is to leave you with
inspectable cached pulses and quick validation plots.

## 1. Create an `Experiment`

Select one qubit and load the configuration that contains its current
pulse parameters.

In [ ]:
import numpy as np

import qubex as qx

exp = qx.Experiment(
    system_id="YOUR_SYSTEM_ID",
    muxes=[0],
    # config_dir="/path/to/qubex-config/config",
    # params_dir="/path/to/qubex-config/params/YOUR_SYSTEM_ID",
)

## 2. Connect to the setup

Connect before you run any checks or pulse calibrations.

In [ ]:
exp.connect()

## 3. Check the waveform

Start with a waveform check so you can catch obvious readout issues before tuning pulse amplitudes.

In [ ]:
waveform_result = exp.check_waveform()

## 4. Measure a baseline Rabi oscillation

Use the initial Rabi measurement as a reference before you update the pulse amplitude.

In [ ]:
rabi_result = exp.obtain_rabi_params()

## 5. Calibrate the default half-pi pulse

Run the half-pi calibration first because many later workflows depend on this pulse.

In [ ]:
hpi_result = exp.calibrate_hpi_pulse(
    targets=exp.qubit_labels,
    n_rotations=1,
)

## 6. Reload and re-check the Rabi oscillation

After updating the control amplitude file, reload and confirm that the Rabi trace reflects the new value.

In [ ]:
# Update `control_amplitude.yaml` manually, then reload
exp.reload()

# Re-check the Rabi oscillation
updated_rabi_result = exp.obtain_rabi_params()

## 7. Plot the calibrated pulse

Plot the cached half-pi pulse for the target qubit so you can inspect the waveform directly.

In [ ]:
Q0 = exp.qubit_labels[0]

hpi_pulse = exp.hpi_pulse[Q0]
hpi_pulse.plot()

## 8. Repeat the calibrated half-pi pulse

Use a repeated-sequence check to confirm that the calibrated pulse accumulates as expected.

In [ ]:
repeat_result = exp.repeat_sequence(
    {Q0: hpi_pulse},
    repetitions=20,
)

## 9. Optionally save the plot

Export the repeated-sequence plot only when you want to keep a record of the result.

In [ ]:
repeat_result.plot(normalize=True, images_dir="./images")